In [1]:
from pathlib import Path

from eyened_orm.importer.importer import Importer
from eyened_orm.importer.importer_dtos import PatientImport, StudyImport, SeriesImport, ImageImport
from eyened_orm import Database


In [2]:
from dotenv import load_dotenv
env_path = Path.cwd().parent.parent / ".env.dev"
load_dotenv(env_path)

session = Database().create_session()

In [3]:
# Set up project information
project_name = "FAU Fundus Dataset"
extract_dir = Path("/mnt/oogergo/eyened/public_data/av_segmentation/HRF_AV")
# Get all image paths
images_dir = extract_dir / "images"
image_paths = list(images_dir.glob("*"))

print(f"Found {len(image_paths)} images.")

Found 45 images.


In [4]:
storage_backends = {
    "/mnt/oogergo/": "oogergo",
}

def split_path(path):
    for key, value in storage_backends.items():
        if str(path).startswith(key):
            return value, str(path)[len(key) :]
    raise ValueError(f"Path {path} not found in storage backends")

def get_image_import(img_path):
    storage_backend_key, object_key = split_path(img_path)
    return ImageImport(object_key=object_key, storage_backend_key=storage_backend_key)


In [5]:
# The dataset has images named as g0001.jpg, g0002.jpg, etc.
# Let's group images together in batches to demonstrate the hierarchy

# Create the data structure for the Importer
# Even though in this dataset each image is of a different patient
# We'll create one "patient" for every 10 images as an example
from datetime import date


data = []
batch_size = 10

for i in range(0, len(image_paths), batch_size):
    batch_images = image_paths[i : i + batch_size]

    # Create image objects
    images = [get_image_import(img_path) for img_path in batch_images]

    # Create hierarchy objects
    series = SeriesImport(images=images)
    study = StudyImport(study_date=date.today(), series=[series])

    patient_item = PatientImport(
        project_name=project_name,
        patient_identifier=f"Patient_{i // batch_size + 1}",
        studies=[study],
    )

    data.append(patient_item)

print(f"Created data structure with {len(data)} patients.")
print(f"First patient has {len(data[0].studies[0].series[0].images)} images.")

Created data structure with 5 patients.
First patient has 10 images.


### Summaries

The summary method will return a summary of created ORM objects without writing anything to the database

In [6]:
# We need to specify create_patients=True, create_studies=True
# since we're not providing identifiers
importer = Importer(
    session=session,
    create_projects=True,
    create_patients=True,
    create_studies=True,
    run_ai_models=False, 
    generate_thumbnails=True,
)

# Display a summary of what will be imported
import_summary = importer.import_many(data, summary=True)


Import Summary for Projects: FAU Fundus Dataset
----------------  Object Statistics  ----------------
        Entity  Total  New  Existing New_Percentage Existing_Percentage
 ImageInstance     45   45         0         100.0%                0.0%
       Project      1    0         1           0.0%              100.0%
       Patient      5    5         0         100.0%                0.0%
   DeviceModel      1    0         1           0.0%              100.0%
StorageBackend      1    0         1           0.0%              100.0%
  ImageStorage     45   45         0         100.0%                0.0%
         Study      5    5         0         100.0%                0.0%
DeviceInstance      1    0         1           0.0%              100.0%
        Series      5    5         0         100.0%                0.0%

-----------  Column Population Statistics  -----------
- only for new entities
- values set to NULL are not considered populated

Populated ImageInstance Columns:
           Co

### Execution

Once satisfied with the summary, commit the import to the database with `importer.exec`
There might still be Exceptions generated during import, in which case nothing will change in the DB and no files will be written.

Post-insertion scripts will run after insertion for non-essential steps such as:

- Thumbnail generation (highly recommended to run)
- Running image preprocessing scripts (eg. CFI bounds detection)
- Running AI models which populate DB columns
- Hashing the files for error and duplicate checks

In [7]:
importer.import_many(data)

[ImageInstance(ImageInstanceID=56, PublicID=16w5xxw8, SeriesID=7, SourceInfoID=None, DeviceInstanceID=2, ModalityID=None, ScanID=None, Modality=None, SOPInstanceUid=None, SOPClassUid=None, PhotometricInterpretation=None, SamplesPerPixel=None, Rows_y=None, Columns_x=None, NrOfFrames=None, SliceThickness=None, ResolutionAxial=None, ResolutionHorizontal=None, ResolutionVertical=None, HorizontalFieldOfView=None, Laterality=None, DICOMModality=None, AnatomicRegion=None, ETDRSField=None, Angiography=None, AcquisitionDateTime=None, PupilDilated=None, DatasetIdentifier=, AltDatasetIdentifier=None, ThumbnailPath=None, OldPath=None, FDAIdentifier=None, Inactive=False, CFROI=None, CFKeypoints=None, CFQuality=None, DateInserted=2026-03-11 11:51:16, DateModified=None, DatePreprocessed=None),
 ImageInstance(ImageInstanceID=67, PublicID=cm4rmvtw, SeriesID=8, SourceInfoID=None, DeviceInstanceID=2, ModalityID=None, ScanID=None, Modality=None, SOPInstanceUid=None, SOPClassUid=None, PhotometricInterpreta

### Inspect Results

The project is now in the DB. We can easily generate a dataframe with image details to inspect results

In [8]:
# inspect the project using the ORM
from eyened_orm import Project

project = Project.by_name(session, 'FAU Fundus Dataset')

In [9]:
project.make_dataframe()

,ImageInstanceID,PublicID,SeriesID,SourceInfoID,DeviceInstanceID,ModalityID,ScanID,Modality,SOPInstanceUid,SOPClassUid,...,ThumbnailPath,OldPath,FDAIdentifier,Inactive,CFROI,CFKeypoints,CFQuality,DateInserted,DateModified,DatePreprocessed
0,46,s0hiq4kt,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
1,47,eowkbmct,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
2,48,k41kgeyv,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
3,49,5qyvlcor,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
4,50,ki40y8q4,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
5,51,iu48bf2i,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
6,52,8euo3ruz,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
7,53,krf90y3k,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
8,54,8pc2rv4t,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None
9,55,4r70uru3,6,None,2,None,None,None,None,None,...,None,None,None,False,None,None,None,2026-03-11 11:51:16,None,None


In [10]:
raise Exception("Stop here")

Exception: Stop here

### Updating existing projects

Images can be inserted into existing projects, patients, studies and series by passing an existing project name, patient_identifier, study_date or series_id in the input structure. These will be matched to database entities (taking into account their nested structure). 

In [ ]:
# to simulate inserting into an existing project, we'll insert the same data again
# use create_patients=False, create_studies=False, create_series=False when inserting into existing objects to ensure that no new objects will be created
# copy_files=True will copy the files to a configurable directory
importer = Importer(
    session=session,
    run_ai_models=True, 
    generate_thumbnails=True,
    # copy_files=True # removed copy_files as it wasn't in __init__ signature I read
)

# Display a summary of what will be imported
import_summary = importer.import_many(data, summary=True)

RuntimeError: Project with name 'FAU Fundus Dataset' not found and create_projects=False

In [ ]:
importer.import_many(data)

100%|██████████| 45/45 [00:00<00:00, 57264.47it/s]


No images to process


[ImageInstance(ImageInstanceID=136, PublicID=a6c6dl5q, SeriesID=106, SourceInfoID=None, DeviceInstanceID=1, ModalityID=None, ScanID=None, Modality=None, SOPInstanceUid=None, SOPClassUid=None, PhotometricInterpretation=None, SamplesPerPixel=None, Rows_y=None, Columns_x=None, NrOfFrames=None, SliceThickness=None, ResolutionAxial=None, ResolutionHorizontal=None, ResolutionVertical=None, HorizontalFieldOfView=None, Laterality=None, DICOMModality=None, AnatomicRegion=None, ETDRSField=None, Angiography=None, AcquisitionDateTime=None, PupilDilated=None, DatasetIdentifier=eyened/public_data/av_segmentation/HRF_AV/images/15_g.jpg, AltDatasetIdentifier=None, ThumbnailPath=, OldPath=None, FDAIdentifier=None, Inactive=False, CFROI=None, CFKeypoints=None, CFQuality=None, DateInserted=2026-03-10 16:48:14, DateModified=2026-03-10 16:48:14, DatePreprocessed=None),
 ImageInstance(ImageInstanceID=137, PublicID=xra9wozv, SeriesID=106, SourceInfoID=None, DeviceInstanceID=1, ModalityID=None, ScanID=None, M

In [ ]:
project = Project.by_name(session, 'FAU Fundus Dataset')
project.make_dataframe()

,ImageInstanceID,PublicID,SeriesID,SourceInfoID,DeviceInstanceID,ModalityID,ScanID,Modality,SOPInstanceUid,SOPClassUid,...,ThumbnailPath,OldPath,FDAIdentifier,Inactive,CFROI,CFKeypoints,CFQuality,DateInserted,DateModified,DatePreprocessed
0,91,texke9in,96,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:47:22,2026-03-10 16:47:22,None
1,92,oepw2871,96,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:47:22,2026-03-10 16:47:22,None
2,93,a4ngali0,96,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:47:22,2026-03-10 16:47:22,None
3,94,w5nm1ell,96,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:47:22,2026-03-10 16:47:22,None
4,95,41aqx44d,96,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:47:22,2026-03-10 16:47:22,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,176,6fq7nldz,110,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:48:14,2026-03-10 16:48:14,None
86,177,lza6yjo8,110,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:48:14,2026-03-10 16:48:14,None
87,178,vfn8fzmz,110,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:48:14,2026-03-10 16:48:14,None
88,179,z188pzxm,110,None,1,None,None,None,None,None,...,,None,None,False,None,None,None,2026-03-10 16:48:14,2026-03-10 16:48:14,None
